In [0]:
%pip install --upgrade gtfs-realtime-bindings

In [0]:
 dbutils.library.restartPython()

Possible <category> values are rapid-bus-kl , rapid-bus-mrtfeeder, rapid-bus-kuantan and rapid-bus-penang.
#GET https://api.data.gov.my/gtfs-realtime/vehicle-position/prasarana?category=<category>

In [0]:
# pip install gtfs-realtime-bindings pandas requests
from google.transit import gtfs_realtime_pb2
from google.protobuf.json_format import MessageToDict
import pandas as pd
from requests import get
 
# Sample GTFS-R URL from Malaysia's Open API
URL = 'https://api.data.gov.my/gtfs-realtime/vehicle-position/prasarana?category=rapid-bus-kl'
 
# Parse the GTFS Realtime feed
feed = gtfs_realtime_pb2.FeedMessage()
response = get(URL)
feed.ParseFromString(response.content)
 
# Extract and print vehicle position information
vehicle_positions = [MessageToDict(entity.vehicle) for entity in feed.entity]
#print(f'Total vehicles: {len(vehicle_positions)}')
df = pd.json_normalize(vehicle_positions)
display(df)



In [0]:


path = "/Volumes/dbacademy/default/default_volume/routes.txt"  # or the file directly
spark_df_routes = (spark.read
      .format("csv")
      .option("header", "true")      # set to "true" if first row is headers
      .option("sep", ",")
      .option("inferSchema", "true")  # optional
      .load(path))
#display(df)

 
spark_df_routes.createOrReplaceTempView("vw_df_rapid_kl_bus_routes")

In [0]:
spark_df = spark.createDataFrame(df)   
spark_df2 = spark_df.toDF(*[c.replace(".", "_") for c in spark_df.columns])
spark_df2.createOrReplaceTempView("vw_df_rapid_kl_bus")

spark.sql("DROP TABLE IF EXISTS dbacademy.default.rapid_kl_bus")

result = spark.sql("""
SELECT timestamp as timestamp,
trip_tripId as tripId,
trip_startTime as startTime,
trip_startDate as startDate,
trip_routeId as routeId,
position_latitude as latitude,
position_longitude as longitude,
position_bearing as bearing,
position_speed as speed,
vehicle_id as vehicle_id,
vehicle_licensePlate as vehicle_licensePlate,
route.agency_id,
route.route_short_name,
route.route_long_name,
route.route_type,
route.route_color,
route.route_text_color
FROM vw_df_rapid_kl_bus bus
LEFT JOIN vw_df_rapid_kl_bus_routes route on bus.trip_routeId = route.route_id
""")

result.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dbacademy.default.rapid_kl_bus")